# U-Net con recortes falsos (#7)
Entrena la U-Net con recortes falsos para comprobar que el pipeline funciona en Colab con GPU.
Para GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU.


In [ ]:
try:
    import google.colab
    COLAB = True
except:
    COLAB = False

RAMA = "7-unet-falsos"   # después del merge, cambiar a "main"

if COLAB:
    !git clone -b {RAMA} https://github.com/sofia2005222/vacamuerta-metano-seg.git
    %cd vacamuerta-metano-seg
    !pip install -qq -e .
else:
    %cd ..
    %load_ext autoreload
    %autoreload 2


In [ ]:
import torch
from torchinfo import summary

from vmseg.config import DatasetConfig, TrainingConfig, deterministic
from vmseg.dataset import get_data
from vmseg.entrenar import main
from vmseg.modelo import crear_unet
from vmseg.recorte import generar_recortes_falsos

deterministic(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

generar_recortes_falsos("data/falsos", n=64)

dataset_config = DatasetConfig(data_root="data/falsos", batch_size=8, augment=True)
train_config = TrainingConfig(epochs=15, learning_rate=1e-3, device=device,
                              checkpoints_dir="runs/falsos")

train_loader, valid_loader, mean, std = get_data(dataset_config)


In [ ]:
model = crear_unet(in_channels=len(dataset_config.bandas)).to(device)
print(summary(model, input_size=(1, 6, 256, 256), depth=2))

optimizer = torch.optim.AdamW(model.parameters(), lr=train_config.learning_rate,
                              weight_decay=train_config.weight_decay)

history = main(model, optimizer, train_loader, valid_loader, train_config)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history["iter_train_loss"])
axes[0].set_title("Loss por iteración (train)"); axes[0].set_xlabel("Iteración")

axes[1].plot(history["train_loss"], "-o", label="train")
axes[1].plot(history["valid_loss"], "-o", label="val")
axes[1].set_title("Loss por época"); axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(history["train_miou"], "-o", label="train")
axes[2].plot(history["valid_miou"], "-o", label="val")
axes[2].set_title("mIoU (locación y pileta)"); axes[2].set_xlabel("Epoch"); axes[2].legend()

for ax in axes: ax.grid(True)
plt.show()
